# 00 · 云账号与安全基线

**一句话**：在往云上放任何东西之前，先把账号锁好、装上报警器。

就像搬进新房子的第一天：

| 新房子 | AWS 账号 | 本节做什么 |
|---|---|---|
| 房产证原件锁进抽屉 | **root 账号**：权限无限，不能日常使用 | 开 MFA，然后基本不再用它 |
| 给自己配一把日常钥匙 | **IAM 用户**：你每天用的身份 | 建 `pipi-admin`，也开 MFA |
| 钥匙 12 小时后自动失效 | **临时凭证**：`aws login` | 本机命令行不存任何长期密钥 |
| 窗户统一上锁 | **S3 禁止公开访问**（账户级） | 一键打开 |
| 烟雾报警器 | **预算报警** | 10 / 25 / 50 美元三档 |
| 门口的监控摄像头 | **CloudTrail**：每个操作都有记录 | 查看你自己刚做过的操作 |

**本节花费：0 美元**（用到的全是免费功能）

**怎么用**：
1. 先在浏览器里完成第 2 部分的手动步骤
2. VS Code 右上角点 **Select Kernel** → **Jupyter Kernel...** → 选 **Python (qafi-ml-platform)**（对应项目里的 `.venv` 环境）
3. 从第 3 部分开始逐格 `Shift+Enter`。如果中途装了新的包，先点工具栏的 **Restart** 重启 kernel，再从 3.1 重新运行

## 1 · 先懂三个概念

### 1.1 共享责任模型（Shared Responsibility Model）

**AWS 负责"云本身"的安全**：机房、硬件、网络、虚拟化。
**你负责"云里面"的安全**：谁能登录、谁有什么权限、数据有没有加密、桶有没有对外公开。

绝大多数云上的安全事故不是 AWS 被攻破，而是**用户自己配错了**：桶设成了公开、密钥被传到了 GitHub、权限给得太大。这一节做的就是"你那一半"的基本功。

### 1.2 三种身份

| 身份 | 比喻 | 什么时候用 |
|---|---|---|
| **root** | 房产证原件 | 几乎从不。只在少数必须由 root 做的事情时用（比如关闭账号） |
| **IAM 用户** | 你的日常钥匙 | 人登录控制台、本机命令行 |
| **IAM 角色**（role） | 临时通行证 | **程序**用：Lambda、Batch 任务、GitHub Actions。没有密码，只发临时凭证 |

### 1.3 长期密钥 vs 临时凭证

- **长期密钥**（access key）= 一张永久门禁卡。写在文件里，一旦泄露（比如不小心提交到 GitHub），别人可以一直用，直到你发现并作废它。GitHub 上每天都有机器人在扫描泄露的 AWS 密钥
- **临时凭证** = 12 小时后自动失效的门禁卡。泄露了，损失也有时间上限

**本项目的规则：本机和 GitHub 上都不存任何长期密钥。** 本机用 `aws login`，GitHub Actions 用 OIDC（02 号讲）。

> **公司里怎么做**：用 **IAM Identity Center**（单点登录），人通过公司账号登录，再"扮演"不同的角色。我们**不用**它，因为 Identity Center 会自动创建 AWS Organization，而 **Free plan 账号一旦加入 Organization，免费额度会立刻作废**。面试时可以讲这个取舍。

## 2 · 手动步骤（在浏览器里做，大约 15 分钟）

这一部分只能你自己做：涉及手机验证、密码这些不能交给代码的东西。**做完一步打一个勾。**

### [ ] 第 1 步：给 root 开 MFA

1. 用注册时的邮箱以 **root** 身份登录 [AWS 控制台](https://console.aws.amazon.com/)
2. 右上角点账号名 → **Security credentials**
3. **Multi-factor authentication (MFA)** → **Assign MFA device**
4. 推荐选 **Passkey or security key**（用 Mac 的 Touch ID），或者 **Authenticator app**（手机上的 Google Authenticator / Microsoft Authenticator）

### [ ] 第 2 步：允许 IAM 用户查看账单

1. 右上角账号名 → **Account**
2. 找到 **IAM user and role access to Billing information** → **Edit** → 勾选 **Activate IAM Access** → **Update**

不做这一步的话，下面建的 `pipi-admin` 即使有管理员权限，也看不到账单和预算。（如果你的控制台里找不到这一项，先跳过，第 3.5 节会自动检查。）

### [ ] 第 3 步：建日常使用的管理员用户

1. 顶部搜索框输入 **IAM** → 左侧 **Users** → **Create user**
2. User name：`pipi-admin`
3. 勾选 **Provide user access to the AWS Management Console**
4. **注意**：如果页面推荐 **"Specify a user in Identity Center – Recommended"**，**不要选它**（原因见 1.3），选 **I want to create an IAM user**
5. 设一个强密码 → **Next**
6. **Attach policies directly** → 搜索并勾选 **AdministratorAccess** → **Next** → **Create user**
7. 记下页面上的 **Console sign-in URL**（形如 `https://123456789012.signin.aws.amazon.com/console`），加入浏览器书签
8. 回到 **Users** → 点 `pipi-admin` → **Security credentials** → **Assign MFA device**，同样开 MFA

> **为什么先给 AdministratorAccess？** 学习阶段我们要用很多服务，一开始就精细授权会卡在权限上。这是**有意识的取舍**：20 号 notebook 会根据 CloudTrail 记录收紧权限（"最小权限原则"）。面试被问到时，"我知道这是取舍、计划了收紧"比"我给了管理员权限"好得多。

### [ ] 第 4 步：退出 root，换成 `pipi-admin`

1. 右上角 → **Sign out**
2. 打开刚才的 Console sign-in URL，用 `pipi-admin` 登录
3. 右上角区域选 **Europe (Frankfurt) eu-central-1**

**从现在起，root 只锁在抽屉里。**

### [ ] 第 5 步：本机命令行登录（在 VS Code 终端里）

```bash
aws login --profile qafi
```

- 提示 `AWS Region [us-east-1]:` 时输入 **`eu-central-1`**
- 浏览器会自动打开，选择 `pipi-admin` 的会话（或用它登录）
- 回到终端看到成功提示后，验证一下：

```bash
aws sts get-caller-identity --profile qafi
```

输出里的 `Arn` 应该以 `user/pipi-admin` 结尾。

> `aws login` 拿到的是**临时凭证**，自动刷新，最长 12 小时；过期后重新运行 `aws login --profile qafi` 即可。用完想主动退出：`aws logout --profile qafi`。

## 3 · 用代码检查和配置

手动部分做完后，从这里开始逐格运行。

### 3.1 连接 AWS

In [ ]:
import os
from datetime import datetime, timedelta, timezone

import boto3

PROFILE = os.environ.get("QAFI_AWS_PROFILE", "qafi")   # 第 5 步 aws login 时用的名字
REGION = "eu-central-1"                                 # 法兰克福：项目的所有资源都放在这里

session = boto3.Session(profile_name=PROFILE, region_name=REGION)
print(f"profile = {PROFILE}, region = {REGION}")

`boto3` 是 AWS 的 Python SDK。**Session** 记住"用哪个身份、在哪个区域"，之后从它创建各个服务的 **client**（客户端）。命令行里 `aws xxx` 能做的事，`boto3` 都能做。

> **一个真实踩过的坑**：`aws login` 的凭证，boto3 要通过 AWS 的一个底层组件 **CRT**（Common Runtime）才能读取。只装 `boto3` 的话，下一格会报 `MissingDependencyException: ... pip install "botocore[crt]"`。所以 `pyproject.toml` 里写的是 `boto3[crt]`。命令行 `aws` 自带这个组件，所以命令行能用、Python 不能用，这种情况就是它。

### 3.2 我是谁？

In [ ]:
identity = session.client("sts").get_caller_identity()
ACCOUNT_ID = identity["Account"]
USER_ARN = identity["Arn"]
USER_NAME = USER_ARN.split("/")[-1]

print("账号 ID :", ACCOUNT_ID)
print("身份    :", USER_ARN)
assert not USER_ARN.endswith(":root"), "你正在用 root！请用 pipi-admin 重新运行 aws login --profile qafi"
print("[OK] 不是 root")

**STS**（Security Token Service）是发临时凭证的服务。`get_caller_identity` 不需要任何权限，是排查"我现在到底用的是哪个身份"的第一条命令。面试里如果被问到"脚本报权限错误你先查什么"，答这一条。

### 3.3 免费额度还剩多少？

In [ ]:
freetier = session.client("freetier", region_name="us-east-1")   # 这个 API 只在 us-east-1
try:
    plan = freetier.get_account_plan_state()
except freetier.exceptions.ResourceNotFoundException:
    plan = None   # 2025 年 7 月以前注册的老账号没有这项信息，不算错误

if plan:
    credits = plan.get("accountPlanRemainingCredits", {})
    print(f"计划类型 : {plan['accountPlanType']}   状态: {plan['accountPlanStatus']}")
    print(f"剩余额度 : {credits.get('amount')} {credits.get('unit')}")
    print(f"到期日   : {plan.get('accountPlanExpirationDate')}")
else:
    print("这个账号没有免费计划信息（老账号）")

以后任何时候想知道"还剩多少钱"，重新运行这一格就行。

**只捕获能处理的错误**：这里只捕获 `ResourceNotFoundException`（老账号没有计划信息），因为我们知道该怎么处理它。其他错误（比如权限不够）不捕获，让它直接报出来。如果把所有错误都"打印一下然后继续"，真正的问题就被藏起来了。

### 3.4 我们要用的服务，法兰克福都有吗？

AWS 的新服务往往先在美国上线，欧洲区域晚一些。动手之前先确认，免得做到第 17 号才发现某个服务在法兰克福用不了。

AWS 把"每个区域有哪些服务"公开放在 **SSM Parameter Store** 里，可以直接查询：

In [ ]:
ssm = session.client("ssm")
paginator = ssm.get_paginator("get_parameters_by_path")   # 一页最多 10 条，必须翻页

pages = paginator.paginate(Path=f"/aws/service/global-infrastructure/regions/{REGION}/services")
available = set(pages.search("Parameters[].Value"))   # 跨所有页提取每条记录的 Value
print(f"{REGION} 一共有 {len(available)} 个服务\n")

NEEDED = {   # 计划里每一层要用的服务 → 它在这份列表里的名字
    "S3": "s3", "Athena": "athena", "Glue": "glue", "Lake Formation": "lakeformation",
    "Batch": "batch", "ECR": "ecr", "ECS / Fargate": "ecs", "Lambda": "lambda",
    "API Gateway": "apigateway", "Step Functions": "stepfunctions", "EventBridge": "eventbridge",
    "SageMaker": "sagemaker", "Bedrock": "bedrock", "Bedrock AgentCore": "bedrock-agentcore",
    "S3 Vectors": "s3vectors", "KMS": "kms", "CloudTrail": "cloudtrail",
    "CloudWatch": "cloudwatch", "SNS": "sns", "Secrets Manager": "secretsmanager",
}
for label, name in NEEDED.items():
    if name in available:
        print(f"[OK] {label}")
    else:
        similar = sorted(s for s in available if name.replace("-", "") in s.replace("-", ""))
        print(f"[??] {label}: 没找到 '{name}'" + (f"，相近的名字: {similar}" if similar else "，需要人工确认"))

**分页（pagination）** 是写云代码时最常见的坑之一：大多数"列出"类的 API 一次只返回一页（这里是 10 条），不翻页就会**悄悄漏掉数据**，不会报错。boto3 的 `get_paginator` 会自动翻完所有页；`.search()` 用 JMESPath 表达式从每一页里提取需要的字段。

如果某个服务显示 `[??]`，把结果发给我，我们再决定是换区域还是换服务。

### 3.5 账户级：禁止 S3 公开访问

S3 桶被意外设成公开，是云上最常见的数据泄露原因。**账户级**的开关会覆盖所有桶，以后哪怕某个桶配错了，也公开不了。

In [ ]:
s3control = session.client("s3control")
s3control.put_public_access_block(
    AccountId=ACCOUNT_ID,
    PublicAccessBlockConfiguration={
        "BlockPublicAcls": True,        # 拒绝新的公开 ACL
        "IgnorePublicAcls": True,       # 忽略已有的公开 ACL
        "BlockPublicPolicy": True,      # 拒绝公开的桶策略
        "RestrictPublicBuckets": True,  # 已经公开的桶也只允许本账号访问
    },
)
pab = s3control.get_public_access_block(AccountId=ACCOUNT_ID)["PublicAccessBlockConfiguration"]
print("账户级 S3 公开访问阻止:", pab)
assert all(pab.values())
print("[OK] 四项全部开启")

这一格**重复运行是安全的**：设成同样的值，结果不变。这就是**幂等**（idempotent），在云上操作时非常重要，因为脚本、流水线随时可能被重跑。

### 3.6 预算报警：最重要的一格

先填你接收报警的邮箱：

In [ ]:
ALERT_EMAIL = ""   # ← 填你的邮箱，例如 "you@example.com"
assert "@" in ALERT_EMAIL, "请先在上面填好邮箱，再运行这一格"

In [ ]:
budgets = session.client("budgets", region_name="us-east-1")   # 预算是全局服务，固定用 us-east-1
BUDGET_NAME = "qafi-monthly-cost"
LIMIT_USD = 50

def email_alert(kind: str, percent: int) -> dict:
    return {
        "Notification": {
            "NotificationType": kind,            # ACTUAL = 已经花了；FORECASTED = 预测本月会花
            "ComparisonOperator": "GREATER_THAN",
            "Threshold": percent,
            "ThresholdType": "PERCENTAGE",
        },
        "Subscribers": [{"SubscriptionType": "EMAIL", "Address": ALERT_EMAIL}],
    }

alerts = [
    email_alert("ACTUAL", 20),       # 已花 10 美元
    email_alert("ACTUAL", 50),       # 已花 25 美元
    email_alert("ACTUAL", 100),      # 已花 50 美元
    email_alert("FORECASTED", 100),  # 预测本月会超过 50 美元：提前提醒
]

budget = {
    "BudgetName": BUDGET_NAME,
    "BudgetLimit": {"Amount": str(LIMIT_USD), "Unit": "USD"},
    "TimeUnit": "MONTHLY",
    "BudgetType": "COST",
    "CostTypes": {"IncludeCredit": False, "IncludeRefund": False},   # ← 关键，见下方说明
}

try:
    budgets.create_budget(AccountId=ACCOUNT_ID, Budget=budget, NotificationsWithSubscribers=alerts)
    print(f"[OK] 预算 {BUDGET_NAME} 已创建：10 / 25 / 50 美元 + 预测超支提醒 -> {ALERT_EMAIL}")
except budgets.exceptions.DuplicateRecordException:
    print(f"预算 {BUDGET_NAME} 已经存在，没有重复创建（幂等）")

### 注意：为什么 `IncludeCredit` 必须是 `False`

默认情况下，预算统计的是**扣掉免费额度之后**的金额。你在用额度的这段时间，这个数字一直是 **0 美元**，报警**永远不会响**。等额度用完，第一笔真实扣费才会让你知道。

设成 `False` 之后，预算统计的是**真实消耗**（不管是额度付还是信用卡付），你花掉额度的速度会实时触发报警。

> **面试细节**：*"I set budgets on gross cost excluding credits; otherwise alerts stay silent until the credits run out."*

### 3.7 审计：你刚才的每一步都被记录了

**CloudTrail** 自动记录账号里的每一个 API 操作：谁、什么时候、从哪里、做了什么。最近 90 天的记录免费保存，不用任何配置。

> 记录会延迟 5–15 分钟才出现。如果下面是空的，过一会儿再运行这一格。

In [ ]:
since = datetime.now(timezone.utc) - timedelta(hours=3)

# 全局服务（预算、IAM）的操作记录在 us-east-1，区域服务（S3 控制等）记录在 eu-central-1
for region in ("us-east-1", REGION):
    trail = session.client("cloudtrail", region_name=region)
    events = trail.lookup_events(
        LookupAttributes=[{"AttributeKey": "Username", "AttributeValue": USER_NAME}],
        StartTime=since,
        MaxResults=20,
    )["Events"]
    print(f"[{region}] 最近 3 小时 {USER_NAME} 的操作：{len(events)} 条")
    for ev in events:
        print(f"   {ev['EventTime']:%H:%M:%S}  {ev['EventSource']:<35} {ev['EventName']}")

面试题 *"有人删了一个生产环境的桶，你怎么查是谁？"* 的标准答案就是这个：在 CloudTrail 里按 `EventName = DeleteBucket` 查，记录里有用户、时间、来源 IP。

### 3.8 安全基线体检

In [ ]:
iam = session.client("iam")
summary = iam.get_account_summary()["SummaryMap"]

def has_budget() -> bool:
    try:
        found = budgets.describe_budgets(AccountId=ACCOUNT_ID).get("Budgets", [])
    except budgets.exceptions.NotFoundException:
        return False
    return any(b["BudgetName"] == BUDGET_NAME for b in found)

checks = {
    "root 账号已开 MFA": summary.get("AccountMFAEnabled") == 1,
    "root 没有长期密钥": summary.get("AccountAccessKeysPresent", 0) == 0,
    "当前身份不是 root": not USER_ARN.endswith(":root"),
    f"{USER_NAME} 已开 MFA": len(iam.list_mfa_devices(UserName=USER_NAME)["MFADevices"]) > 0,
    "账户级禁止 S3 公开访问": all(s3control.get_public_access_block(AccountId=ACCOUNT_ID)["PublicAccessBlockConfiguration"].values()),
    "预算报警已设置": has_budget(),
}

for item, ok in checks.items():
    print("[OK]  " if ok else "[FAIL]", item)
print("\n全部通过" if all(checks.values()) else "\n有 [FAIL] 的项：回到第 2 部分对应的步骤补上，然后重新运行这一格")

这一格就是一个小型的**合规检查**（compliance check）：把"安全要求"写成可以自动运行的代码，任何时候都能重新检查一遍。公司里有专门的服务做这件事（AWS Config、Security Hub），原理一样。

---

## 4 · 本节小结

| 做了什么 | 为什么 |
|---|---|
| root 开 MFA，然后不再使用 | root 权限无限，一旦泄露损失无上限 |
| 建 `pipi-admin` + MFA | 日常身份，权限可以被收紧、可以被审计 |
| `aws login` 临时凭证 | 本机不存长期密钥 |
| 账户级禁止 S3 公开 | 云上最常见的泄露原因，从根上堵住 |
| 预算报警（不扣额度） | 花钱的速度实时可见 |
| CloudTrail | 每个操作都可追溯 |

下一节 **01 · Terraform**：用代码创建项目的第一批资源（4 个 S3 桶 + 加密），并学会一条命令建好、一条命令删掉。

```bash
cd ~/Projects/QAFI_Paper/QAFI_ALL/qafi-ml-platform
git add notebooks/A_foundation/00_account_security_baseline.ipynb
git commit -m "feat: 00 account security baseline"
```

---

## 5 · 面试题（先用英文自己说，再看下面的参考答案）

1. What is the AWS shared responsibility model?
2. Why shouldn't you use the root user day to day? What do you do with it?
3. IAM users, IAM roles, IAM Identity Center: what's the difference, and what do companies use?
4. Why avoid long-term access keys? What do you use instead locally and in CI/CD?
5. How do you prevent surprise cloud bills?
6. Someone deleted a production S3 bucket. How do you find out who?
7. What is S3 Block Public Access, and why enable it at the account level?

### 参考答案

**1. Shared responsibility.**
> "AWS is responsible for security *of* the cloud — data centers, hardware, the virtualization layer. The customer is responsible for security *in* the cloud: identities and permissions, network configuration, encryption, and what's exposed publicly. Most cloud breaches are customer misconfigurations, like a public bucket or a leaked key, so that's where I focus first."

**2. Root user.**
> "Root has unrestricted access that IAM policies can't limit, so it should be locked down: MFA on, no access keys, credentials stored securely, and used only for the few tasks that require it, like changing the support plan or closing the account. Day-to-day work happens through IAM identities that can be scoped and audited."

**3. Users, roles, Identity Center.**
> "An IAM user is a long-lived identity with a password or keys. A role has no credentials of its own: whoever assumes it gets temporary credentials, which is how services like Lambda or a CI pipeline get permissions. Companies use IAM Identity Center so people sign in once with the corporate identity provider and assume roles across many accounts. In my personal project I used an IAM user with MFA and `aws login` instead, because Identity Center requires AWS Organizations, which would have ended the free-plan credits — a conscious trade-off."

**4. Long-term keys.**
> "Static keys don't expire, so a leak — say a key committed to GitHub — stays exploitable until someone notices and rotates it. Locally I use `aws login` or SSO, which issue short-lived credentials. In CI/CD I use OIDC federation: the pipeline exchanges its identity token for a role's temporary credentials, so no secret is stored in the CI system at all."

**5. Surprise bills.**
> "Budgets with actual *and* forecast alerts, tracking gross cost excluding credits so the alerts aren't silenced while credits are being consumed. Cost allocation tags on every resource so I can see which component costs what. Pay-per-use services by default instead of always-on clusters. And infrastructure as code, so anything I create for an experiment can be destroyed with one command."

**6. Who deleted the bucket?**
> "CloudTrail. Look up the `DeleteBucket` event — event history covers 90 days of management events by default — and it shows the identity, the time, the source IP and the request parameters. For longer retention or object-level events, a trail delivers logs to S3, where they can be queried with Athena."

**7. S3 Block Public Access.**
> "It's a set of four guardrails that override bucket policies and ACLs that would grant public access. At the account level it applies to every existing and future bucket, so a single misconfigured bucket policy can't expose data. If something truly must be public, like a static website, I'd serve it through CloudFront with origin access control instead of opening the bucket."